In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# --- LLM CONFIGURATION ---
EXAMPLES_ENABLED = False    # 3-shot examples
REASONING_ENABLED = False   # Chain-of-thought reasoning
CONFIDENCE_ENABLED = False  # Confidence score
ROLE_NR = 1                 # System prompt choice: 1, 2, 3, or 4
MODEL = "GPT5"

# --- DIRECTORIES ---
DIR_CLASSIFICATION = "../classification/classifications"
DIR_SENTIMENT      = "../sentiment analysis"
DIR_READABILITY    = "../readability analysis"
DIR_TRANSLATION    = "."
DIR_LSEG           = "."
DIR_OUTPUT         = "."

# --- FILENAMES ---
# This matches the naming convention used in my analysis scripts
suffix = f"_{MODEL}_role{ROLE_NR}{'_examples' if EXAMPLES_ENABLED else ''}{'_reasoning' if REASONING_ENABLED else ''}{'_confidence' if CONFIDENCE_ENABLED else ''}"

FILE_CLASS = f"PoC classification{suffix}.csv"
FILE_SENT  = f"PoC sentiment{suffix}.csv"
FILE_READ  = f"PoC readability{suffix}.csv"

OUTPUT_FILE = f"regression input{suffix}.csv"

In [5]:
def main():
    print("--- Starting Aggregation Process ---")
    
    # Step 1: Loading input files
    try:
        df_class = pd.read_csv(os.path.join(DIR_CLASSIFICATION, FILE_CLASS))
        df_sent  = pd.read_csv(os.path.join(DIR_SENTIMENT, FILE_SENT))
        df_read  = pd.read_csv(os.path.join(DIR_READABILITY, FILE_READ))
        df_trans = pd.read_csv(os.path.join(DIR_TRANSLATION, "translation_table.csv"))
        df_lseg  = pd.read_csv(os.path.join(DIR_LSEG, "LSEG data.csv"))
    except Exception as e:
        print(f"Error loading files: {e}")
        return

    # Step 2: Combine all data at post-level
    df_posts = df_class.copy()
    df_posts['Year_Extracted'] = df_posts['Date'].astype(str).str[:4]
    
    # Robust sentiment conversion
    # Convert to numeric first (1.0, 0.0, -1.0). NaNs will be handled separately.
    df_posts['sentiment_numeric'] = pd.to_numeric(df_sent['sentiment'], errors='coerce')
    
    # Diagnostic: check what sentiments are actually there
    found_sentiments = df_posts['sentiment_numeric'].unique()
    print(f"Debug: Unique sentiment values found: {found_sentiments}")
    
    # Readability columns
    readability_averages = ['Average_Main_Crit', 'Average_Subcrit', 'Weighted_Main_Crit', 'Weighted_Subcrit']
    read_cols_to_avg = ['word_count', 'flesch_reading_ease', 'gunning_fog']
    dynamic_read_cols = [c for c in df_read.columns if c.endswith('_score')]
    all_read_cols = list(set(read_cols_to_avg + dynamic_read_cols + readability_averages))
    
    for col in all_read_cols:
        if col in df_read.columns:
            df_posts[col] = pd.to_numeric(df_read[col], errors='coerce')

    # Step 3: Mapping & Fuzzy Matching (RIC)
    df_trans['Company'] = df_trans['Company'].astype(str).str.strip()
    df_posts['Company'] = df_posts['Company'].astype(str).str.strip()
    name_to_ric = dict(zip(df_trans['Company'], df_trans['Identifier (RIC)']))
    df_posts['RIC'] = df_posts['Company'].map(name_to_ric)

    def fuzzy_match_ric(row):
        if pd.notna(row['RIC']) and str(row['RIC']) != 'nan': return row['RIC']
        parts = str(row['Company']).split()
        short_name = " ".join(parts[:-1]).lower() if len(parts) > 1 else str(row['Company']).lower()
        for trans_name, ric in name_to_ric.items():
            if trans_name.lower().startswith(short_name): return ric
        return np.nan

    df_posts['RIC'] = df_posts.apply(fuzzy_match_ric, axis=1)
    df_posts['primary_key'] = df_posts['RIC'].astype(str) + "_" + df_posts['Year_Extracted']

    # Step 4: Identify Categories
    for cat in ['Cat_E', 'Cat_S', 'Cat_G', 'Cat_N']:
        df_posts[cat] = df_posts[cat].astype(str).str.strip() if cat in df_posts.columns else "No"

    df_posts['is_esg'] = (df_posts['Cat_E'] == 'Yes') | (df_posts['Cat_S'] == 'Yes') | (df_posts['Cat_G'] == 'Yes')

    # Step 5: Master Aggregation Function
    def get_aggregated_metrics(df_group, prefix):
        d = {}
        count_total = len(df_group)
        d[f'{prefix}_post_count'] = count_total
        
        # Averages
        d[f'{prefix}_sentiment_avg'] = df_group['sentiment_numeric'].mean()
        for col in all_read_cols:
            if col in df_group.columns:
                d[f'{prefix}_{col}_avg'] = df_group[col].mean()

        # Sentiment Counts (Using numeric comparison for accuracy)
        d[f'{prefix}_sentiment_pos_1_count'] = (df_group['sentiment_numeric'] == 1).sum()
        d[f'{prefix}_sentiment_neu_0_count'] = (df_group['sentiment_numeric'] == 0).sum()
        d[f'{prefix}_sentiment_neg_m1_count'] = (df_group['sentiment_numeric'] == -1).sum()
        d[f'{prefix}_sentiment_na_count'] = df_group['sentiment_numeric'].isna().sum()
        
        # Relative Sentiment (1 compared to total posts)
        if count_total > 0:
            d[f'{prefix}_sentiment_1_relative'] = d[f'{prefix}_sentiment_pos_1_count'] / count_total
        else:
            d[f'{prefix}_sentiment_1_relative'] = np.nan

        # ESG Counts
        d[f'{prefix}_esg_count'] = df_group['is_esg'].sum()
        for cat in ['Cat_E', 'Cat_S', 'Cat_G', 'Cat_N']:
            d[f'{prefix}_{cat}_count'] = (df_group[cat] == 'Yes').sum()
        
        # Relative ESG
        if count_total > 0:
            d[f'{prefix}_esg_relative'] = d[f'{prefix}_esg_count'] / count_total
            for cat in ['Cat_E', 'Cat_S', 'Cat_G', 'Cat_N']:
                d[f'{prefix}_{cat}_relative'] = d[f'{prefix}_{cat}_count'] / count_total
        
        return pd.Series(d)

    print("Aggregating data...")
    df_tot_agg = df_posts.groupby('primary_key').apply(lambda x: get_aggregated_metrics(x, "tot")).reset_index()
    
    df_esg_only = df_posts[df_posts['is_esg']].copy()
    df_esg_agg = df_esg_only.groupby('primary_key').apply(lambda x: get_aggregated_metrics(x, "esg")).reset_index()

    # Step 6: Merge Aggregates and LSEG
    df_final = pd.merge(df_tot_agg, df_esg_agg, on='primary_key', how='left')
    
    df_lseg['Primary Key'] = df_lseg['Primary Key'].astype(str).str.strip()
    df_lseg = df_lseg.set_index('Primary Key')
    
    df_final = df_final.set_index('primary_key').join(df_lseg, how='inner')

    # Step 7: ESG-washing calculations
    esg_score = pd.to_numeric(df_final['ESG Score'], errors='coerce')
    esgc_score = pd.to_numeric(df_final['ESG Controversies Score'], errors='coerce')
    df_final['ESG_washing_score'] = np.where(esgc_score > esg_score, 0, esg_score - esgc_score)
    df_final['relative_ESG_washing_score'] = df_final['ESG_washing_score'] / esg_score

    # Step 8: Save
    output_path = os.path.join(DIR_OUTPUT, OUTPUT_FILE)
    df_final.reset_index().to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Success! Saved to {output_path}")

if __name__ == "__main__":
    main()

--- Starting Aggregation Process ---
Debug: Unique sentiment values found: [ 1. nan]
Aggregating data...
Success! Saved to .\regression input_GPT5_role1.csv


C:\Users\kover\AppData\Local\Temp\ipykernel_37820\548341398.py:98: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_tot_agg = df_posts.groupby('primary_key').apply(lambda x: get_aggregated_metrics(x, "tot")).reset_index()
C:\Users\kover\AppData\Local\Temp\ipykernel_37820\548341398.py:101: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_esg_agg = df_esg_only.groupby('primary_key').apply(lambda x: get_aggregated_metri